# GEAR Metal V4 — Full multimodal + graph + site-hypergraph insight run

This notebook is the all-in metal-binding experiment:

DevSet1010 → representation → fusion → residue graph → full candidate-site hypergraph → lock → TestSet300/TestSetNew46.

Primary selection metric: residue-level Dev CV AUPRC.

It tests:
- ESM + ATOM + CHEM + MULTI2 historical anchor
- + ESM-IF
- + SITECTX32
- + ESM-IF + SITECTX32
- Concat / Projected / Gated / Attention / latent metal MoE
- residual rich and donor-focused residue graph
- full trainable residue↔candidate-metal-site hypergraph
- automatic gain decomposition and scientific insight report

No TestSet300/TestSetNew46 result is used for model selection.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, math, time, shutil, subprocess, platform, copy
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
ULTIMATE_DIR = BASE/'GEAR_ULTIMATE'
BASELINE_WORK = BASE/'baseline_reproduction_master_v6'
REPOS = BASELINE_WORK/'repos'

if not ULTIMATE_DIR.exists():
    raise FileNotFoundError(f'{ULTIMATE_DIR} missing')

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(ULTIMATE_DIR/'requirements.txt')
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'biopython', 'scikit-learn', 'tqdm'
])

sys.path.insert(0, str(ULTIMATE_DIR))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, matthews_corrcoef,
    precision_score, recall_score
)

from gear_ultimate import __version__ as GEAR_VERSION
from gear_ultimate import gear_core as core
from gear_ultimate import gear_structure as structure
from gear_ultimate import gear_mechanistic as mech
from gear_ultimate import gear_graph as graph
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Use a Colab GPU runtime for V3.'

RUN_ROOT = BASE/'results'/'GEAR_METAL_V4_FULL_INSIGHT_HYPERGRAPH'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

V3_CACHE = BASE/'cache'/'gear_metal_v3'
V3_CACHE.mkdir(parents=True, exist_ok=True)

print('GEAR:', GEAR_VERSION)
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('RUN_ROOT:', RUN_ROOT)


Mounted at /content/drive
GEAR: 1.0.0
Torch: 2.11.0+cu128
GPU: Tesla T4
RUN_ROOT: /content/drive/MyDrive/Protein-binding-Phi/results/GEAR_METAL_V4_FULL_INSIGHT_HYPERGRAPH


In [2]:
PREPARE_CONTEXT = True
ESM_BATCH_SIZE = 8

RUN_ESMIF = True
RUN_SITECTX = True

RUN_REPRESENTATION_SCREEN = True
RUN_FUSION_SCREEN = True
RUN_GRAPH_SCREEN = True

# The full hypergraph is intentionally delayed until SITECTX32 proves
# that higher-order donor-site context adds Dev signal.
RUN_FULL_SITE_HYPERGRAPH = True

RUN_FINAL_EXTERNAL = True

FORCE_REBUILD_ESMIF = False
FORCE_REBUILD_SITECTX = False

MIN_MEAN_AUPRC_GAIN = 0.002

CUSTOM_MAX_EPOCHS = 45
CUSTOM_PATIENCE = 7
CUSTOM_BATCH = 4096
CUSTOM_LR = 3e-4
CUSTOM_WEIGHT_DECAY = 1e-4
CUSTOM_SEED = 2026

print('Device:', DEVICE)

RUN_HYPERGRAPH_SCREEN = True

HG_RADIUS_A = 7.5
HG_MAX_DONORS = 6
HG_PROTEIN_BATCH = 8
HG_MAX_EPOCHS = 35
HG_PATIENCE = 6
HG_LR = 3e-4
HG_WEIGHT_DECAY = 1e-4
HG_LATENT = 192

GLOBAL_COMPLEXITY_MARGIN = 0.002

print('Full hypergraph:', RUN_HYPERGRAPH_SCREEN)
print('Global complexity margin:', GLOBAL_COMPLEXITY_MARGIN)


Device: cuda
Full hypergraph: True
Global complexity margin: 0.002


In [3]:
ctx = suite.prepare_context(
    base=BASE,
    run_version='metal_v3_context',
    models=('MLP','KAN','XGBoost'),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=ESM_BATCH_SIZE,
    device=DEVICE,
)

display(suite.audit_context(ctx))
display(reporting.dimension_table(ctx.stores['train']))

MULTI2 = ['METALSPEC','ELEC3D','TOPO','SOLV','SEQMOTIF']
ANCHOR_BLOCKS = ['ESM','ATOM','CHEM',*MULTI2]
print('Historical anchor:', ANCHOR_BLOCKS)


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

,split,proteins,residues,positives,prevalence
0,train,1010,170181,2370,0.013926
1,test300,300,62689,881,0.014054
2,test46,46,6227,77,0.012366


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
3,LOCAL,8
10,METALSPEC,18


Historical anchor: ['ESM', 'ATOM', 'CHEM', 'METALSPEC', 'ELEC3D', 'TOPO', 'SOLV', 'SEQMOTIF']


In [5]:
# ============================================================
# 3. DATASETS + OFFICIAL DevSet1010 FOLDS
# ============================================================
TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'


def parse_label(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)


def load_metal_df(path):
    df = pd.read_csv(path)

    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)
    df['metal_binding'] = df['metal_binding'].apply(parse_label)

    if not df.prot_id.is_unique:
        raise RuntimeError(
            f'{path}: duplicate prot_id'
        )

    for r in df.itertuples():
        if len(r.sequence) != len(r.metal_binding):
            raise RuntimeError(
                f'{r.prot_id}: '
                f'sequence={len(r.sequence)} '
                f'label={len(r.metal_binding)}'
            )

    return df


# ============================================================
# Load raw project datasets
# ============================================================

raw_train_df = load_metal_df(TRAIN_CSV)

test300_df = load_metal_df(TEST300_CSV)
test46_df = load_metal_df(TEST46_CSV)

assert len(test300_df) == 300, len(test300_df)
assert len(test46_df) == 46, len(test46_df)


# ============================================================
# Official bindNode24 folds define the actual DevSet1010
# ============================================================

BN24 = REPOS/'bindnode24'

fold_root = (
    BN24 /
    'dataset' /
    'cv_splits' /
    'predicted_structures'
)

if not fold_root.exists():
    raise FileNotFoundError(
        f'Official bindNode24 fold directory missing: '
        f'{fold_root}'
    )


OFFICIAL_FOLDS = {}

official_order = []
seen = set()


for k in range(1, 6):

    fp = fold_root/f'ids_split{k}.txt'

    if not fp.exists():
        raise FileNotFoundError(fp)

    ids = [
        x.strip()
        for x in fp.read_text().splitlines()
        if x.strip()
    ]

    overlap = seen.intersection(ids)

    if overlap:
        raise RuntimeError(
            f'Fold overlap in fold {k}: '
            f'{sorted(overlap)[:10]}'
        )

    OFFICIAL_FOLDS[k] = ids

    official_order.extend(ids)

    seen.update(ids)


official_ids = set(official_order)

raw_ids = set(
    raw_train_df.prot_id.astype(str)
)


# Every official DevSet1010 protein must exist
# in our project CSV.
missing = official_ids - raw_ids

if missing:
    raise RuntimeError(
        'Official DevSet proteins missing from '
        'full_grouped_train_binding_sites_df.csv: '
        f'{sorted(missing)[:20]}'
    )


# ============================================================
# Filter raw Dev lineage -> exact official DevSet1010
# and preserve official fold ordering
# ============================================================

train_lookup = (
    raw_train_df
    .set_index('prot_id', drop=False)
)

train_df = (
    train_lookup
    .loc[official_order]
    .reset_index(drop=True)
)


assert train_df.prot_id.is_unique

assert len(train_df) == 1010, (
    f'Official fold universe contains '
    f'{len(train_df)} proteins, not 1010'
)


# Proteins intentionally excluded from DevSet1010
excluded_ids = sorted(
    raw_ids - official_ids
)


# ============================================================
# Final dataset objects used by ALL downstream V4 experiments
# ============================================================

SPLIT_DF = {
    'DevSet1010': train_df,
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}


SPLIT_STORE_KEY = {
    'DevSet1010': 'train',
    'TestSet300': 'test300',
    'TestSetNew46': 'test46',
}


# ============================================================
# Audit
# ============================================================

print('=' * 80)

print('DATASET AUDIT')

print('=' * 80)

print(
    'Raw training CSV       :',
    len(raw_train_df)
)

print(
    'Official DevSet1010    :',
    len(train_df)
)

print(
    'Excluded from Dev1010  :',
    len(excluded_ids)
)

print(
    'Excluded IDs           :',
    excluded_ids
)

print(
    'TestSet300             :',
    len(test300_df)
)

print(
    'TestSetNew46           :',
    len(test46_df)
)

print(
    'Fold sizes             :',
    {
        k: len(v)
        for k, v in OFFICIAL_FOLDS.items()
    }
)

print('=' * 80)

DATASET AUDIT
Raw training CSV       : 1014
Official DevSet1010    : 1010
Excluded from Dev1010  : 4
Excluded IDs           : ['C8BD48', 'P84801', 'Q00277', 'Q9NZV6']
TestSet300             : 300
TestSetNew46           : 46
Fold sizes             : {1: 202, 2: 203, 3: 203, 4: 202, 5: 200}


In [6]:
def _store_mapping(store):
    if isinstance(store, dict):
        return store
    for attr in ('records','proteins','data','items_by_id'):
        obj=getattr(store,attr,None)
        if isinstance(obj,dict):
            return obj
    raise TypeError(f'Unsupported store type: {type(store)}')

def _record_feature_container(rec):
    if isinstance(rec,dict):
        if isinstance(rec.get('features'),dict):
            return rec['features']
        return rec
    for attr in ('features','feature_dict','blocks'):
        obj=getattr(rec,attr,None)
        if isinstance(obj,dict):
            return obj
    return None

def get_block(store,pid,block):
    mp=_store_mapping(store)
    rec=mp[str(pid)] if str(pid) in mp else mp[pid]
    cont=_record_feature_container(rec)
    if cont is not None and block in cont:
        return np.asarray(cont[block])
    if isinstance(rec,dict) and block in rec:
        return np.asarray(rec[block])
    if hasattr(rec,block):
        return np.asarray(getattr(rec,block))
    raise KeyError(f'{pid}: missing block {block}')

def set_block(store,pid,block,value):
    mp=_store_mapping(store)
    rec=mp[str(pid)] if str(pid) in mp else mp[pid]
    value=np.asarray(value,dtype=np.float32)
    cont=_record_feature_container(rec)
    if cont is not None:
        cont[block]=value
        return
    if isinstance(rec,dict):
        rec[block]=value
        return
    setattr(rec,block,value)

pid0=str(train_df.iloc[0].prot_id)
for b in ANCHOR_BLOCKS:
    a=get_block(ctx.stores['train'],pid0,b)
    print(pid0,b,a.shape,a.dtype)


Q5LL55 ESM (76, 1280) float16
Q5LL55 ATOM (76, 15) float32
Q5LL55 CHEM (76, 24) float32
Q5LL55 METALSPEC (76, 18) float32
Q5LL55 ELEC3D (76, 8) float32
Q5LL55 TOPO (76, 12) float32
Q5LL55 SOLV (76, 10) float32
Q5LL55 SEQMOTIF (76, 10) float32


# Part A — ESM-IF

The ESM-IF branch reuses the exact public ESM-IF1 model used by ESMBind:
`esm_if1_gvp4_t16_142M_UR50`.

Cache behavior:
- TestSet300/TestSetNew46 first reuse the ESMBind feature cache from the completed baseline reproduction.
- DevSet1010 uses the bindNode24 AlphaFold structures.
- All structures are normalized to exact canonical sequence, chain `A`.
- Missing ESM-IF arrays are generated once and persisted.


In [ ]:
from copy import deepcopy
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.SeqUtils import seq1

ESMIF_ROOT = V3_CACHE/'esmif_v1'
ESMIF_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_ESMIF = {
    'TestSet300': BASELINE_WORK/'cache'/'esmbind_TestSet300'/'esm_if',
    'TestSetNew46': BASELINE_WORK/'cache'/'esmbind_TestSetNew46'/'esm_if',
}

AA3 = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
}

def valid_npy(path,shape):
    path=Path(path)
    if not path.exists():
        return False
    try:
        a=np.load(path,mmap_mode='r')
        return a.shape==shape and np.all(np.isfinite(a))
    except Exception:
        return False

def chain_sequence(chain):
    residues=[
        r for r in chain
        if r.id[0]==' ' and r.resname.strip().upper() in AA3
    ]
    seq=''.join(seq1(r.resname.strip().upper()) for r in residues)
    return residues,seq

def source_pdb(split,pid):
    pid=str(pid)
    candidates=[]
    if split=='DevSet1010':
        candidates.append(BN24/'dataset'/'AlphaFold_predictions'/f'{pid}.pdb')
    else:
        candidates.append(
            BASELINE_WORK/'cache'/'metal_external_normalized_structures'/split/f'{pid}.pdb'
        )
        candidates.append(BN24/'dataset'/'AlphaFold_predictions'/f'{pid}.pdb')

    for root in [BASE/'esmFold_pdb_files',BASE/'experimental_pdb',BASE/'pdb']:
        candidates.append(root/f'{pid}.pdb')

    for p in candidates:
        if p.exists():
            return p

    for root in [BASE/'esmFold_pdb_files',BASE/'experimental_pdb']:
        if root.exists():
            hits=list(root.rglob(f'{pid}.pdb'))
            if hits:
                return hits[0]
    return None

def normalize_exact_chain_A(src,target_seq,dst):
    st=PDBParser(QUIET=True).get_structure('src',str(src))
    model=next(st.get_models())
    exact=[]
    diag=[]
    for ch in model:
        residues,seq=chain_sequence(ch)
        diag.append((str(ch.id),len(seq)))
        if seq==target_seq:
            exact.append((ch,residues))
    if not exact:
        raise RuntimeError(f'{src}: no exact chain match; {diag}')
    exact=sorted(exact,key=lambda x:(0 if str(x[0].id)=='A' else 1,str(x[0].id)))
    _,residues=exact[0]

    out_st=Structure('norm')
    out_model=Model(0)
    out_chain=Chain('A')
    out_st.add(out_model)
    out_model.add(out_chain)

    for i,res in enumerate(residues,start=1):
        rr=deepcopy(res)
        rr.detach_parent()
        rr.id=(' ',i,' ')
        out_chain.add(rr)

    dst=Path(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)
    io=PDBIO()
    io.set_structure(out_st)
    io.save(str(dst))

    st2=PDBParser(QUIET=True).get_structure('check',str(dst))
    _,seq2=chain_sequence(next(st2.get_models())['A'])
    if seq2!=target_seq:
        raise RuntimeError(f'{dst}: normalized sequence mismatch')

PDB_NORM_ROOT=ESMIF_ROOT/'pdb'
for split,df in SPLIT_DF.items():
    d=PDB_NORM_ROOT/split
    d.mkdir(parents=True,exist_ok=True)
    for r in df.itertuples():
        dst=d/f'{r.prot_id}.pdb'
        if dst.exists() and not FORCE_REBUILD_ESMIF:
            try:
                st=PDBParser(QUIET=True).get_structure('x',str(dst))
                _,seq=chain_sequence(next(st.get_models())['A'])
                if seq==r.sequence:
                    continue
            except Exception:
                pass

        src=source_pdb(split,r.prot_id)
        if src is None:
            raise FileNotFoundError(f'{split}/{r.prot_id}: source PDB missing')
        normalize_exact_chain_A(src,r.sequence,dst)

print('Exact chain-A PDB normalization PASS')


In [ ]:
ESMIF_PATHS={}
for split,df in SPLIT_DF.items():
    outdir=ESMIF_ROOT/split/'esm_if'
    outdir.mkdir(parents=True,exist_ok=True)
    ESMIF_PATHS[split]=outdir

    srcdir=BASELINE_ESMIF.get(split)
    if srcdir is not None and srcdir.exists() and not FORCE_REBUILD_ESMIF:
        copied=0
        for r in df.itertuples():
            src=srcdir/f'{r.prot_id}.npy'
            dst=outdir/f'{r.prot_id}.npy'
            if (
                not valid_npy(dst,(len(r.sequence),512))
                and valid_npy(src,(len(r.sequence),512))
            ):
                shutil.copy2(src,dst)
                copied+=1
        print(split,'reused baseline ESM-IF:',copied)

missing={}
for split,df in SPLIT_DF.items():
    bad=[
        str(r.prot_id) for r in df.itertuples()
        if not valid_npy(
            ESMIF_PATHS[split]/f'{r.prot_id}.npy',
            (len(r.sequence),512)
        )
    ]
    missing[split]=bad
    print(split,'missing ESM-IF:',len(bad))

NEED_ESMIF_RUNTIME=RUN_ESMIF and any(len(v)>0 for v in missing.values())


## ESM-IF runtime

The next cell only creates the historical-compatible ESMBind environment if ESM-IF files are actually missing.
Existing `.npy` files are skipped protein-by-protein.


In [ ]:
MAMBA_ROOT=Path('/content/micromamba_gear_v3')
MAMBA_ROOT.mkdir(parents=True,exist_ok=True)

def sh(cmd,cwd=None,check=True,capture=False):
    print('+',cmd)
    return subprocess.run(
        cmd,shell=True,cwd=cwd,check=check,text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None
    )

def ensure_micromamba():
    mm=Path('/usr/local/bin/micromamba')
    if mm.exists():
        return
    sh(
        'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest '
        '| tar -xvj -C /tmp bin/micromamba'
    )
    sh('cp /tmp/bin/micromamba /usr/local/bin/micromamba')
    sh('chmod +x /usr/local/bin/micromamba')

def mex(cmd,cwd=None,check=True,capture=False):
    ensure_micromamba()
    prefix=f'MAMBA_ROOT_PREFIX="{MAMBA_ROOT}" micromamba run -n esmbind_v3'
    return sh(f'{prefix} {cmd}',cwd=cwd,check=check,capture=capture)

ESMB=REPOS/'esmbind'
ESMB_SHA='5d81fb5b3965f973bcbbdde443132b31bef07348'

if NEED_ESMIF_RUNTIME:
    ensure_micromamba()
    envdir=MAMBA_ROOT/'envs'/'esmbind_v3'

    if not envdir.exists():
        sh(
            f'MAMBA_ROOT_PREFIX="{MAMBA_ROOT}" '
            'micromamba create -y -n esmbind_v3 -c conda-forge python=3.10 pip'
        )

    if not ESMB.exists():
        sh(f'git clone https://github.com/Structurebiology-BNL/ESMBind.git "{ESMB}"')

    if not (ESMB/'.git').exists():
        raise RuntimeError(f'Not a git repo: {ESMB}')

    sh('git fetch --all --tags',cwd=ESMB)
    sh(f'git reset --hard {ESMB_SHA}',cwd=ESMB)

    mex('python -m pip install -q --upgrade "pip<25" setuptools wheel')
    mex(
        'python -m pip install -q '
        '"numpy<2" scipy biopython tqdm ml-collections '
        '"biotite==0.41.2" "fair-esm==2.0.0"'
    )
    mex(
        'python -m pip install -q '
        '--index-url https://download.pytorch.org/whl/cu121 '
        '"torch==2.4.1"'
    )
    mex(
        'python -m pip install -q '
        '-f https://data.pyg.org/whl/torch-2.4.0+cu121.html '
        '"torch-scatter==2.1.2"'
    )
    mex('python -m pip install -q "torch-geometric==2.6.1"')

    patch=mex(
        r'''python - <<'PY'
from pathlib import Path
import esm.inverse_folding.util as util

p=Path(util.__file__)
text=p.read_text()
marker='# ESMIF_DEVICE_PATCH_GEAR_V3'

if marker not in text:
    lines=[
        '', '', marker,
        'def get_encoder_output(model, alphabet, coords):',
        '    device = next(model.parameters()).device',
        '    batch_converter = CoordBatchConverter(alphabet)',
        '    batch = [(coords, None, None)]',
        '    coords, confidence, _, _, padding_mask = batch_converter(batch, device=device)',
        '    encoder_out = model.encoder.forward(',
        '        coords, padding_mask, confidence, return_all_hiddens=False',
        '    )',
        "    return encoder_out['encoder_out'][0][1:-1, 0]",
        '',
    ]
    p.write_text(text+'\n'.join(lines))
    print('patched',p)
else:
    print('already patched',p)
PY''',
        check=False,capture=True
    )
    print(patch.stdout)
    if patch.returncode!=0:
        raise RuntimeError('ESM-IF device patch failed')

    smoke=mex(
        r'''python - <<'PY'
import torch, esm
import esm.inverse_folding
print('torch',torch.__version__,'cuda',torch.version.cuda,'available',torch.cuda.is_available())
assert torch.cuda.is_available()
m,a=esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
m=m.eval().cuda()
print('ESM-IF model load PASS')
PY''',
        check=False,capture=True
    )
    print(smoke.stdout)
    if smoke.returncode!=0:
        raise RuntimeError('ESM-IF environment smoke test failed')
else:
    print('ESM-IF runtime SKIP: all arrays already cached.')


In [ ]:
def write_fasta(df,path):
    with open(path,'w') as f:
        for r in df.itertuples():
            f.write(f'>sp|{r.prot_id}|{r.prot_id}\n{r.sequence}\n')

if RUN_ESMIF and NEED_ESMIF_RUNTIME:
    for split,df in SPLIT_DF.items():
        outdir=ESMIF_PATHS[split]

        still_missing=[
            str(r.prot_id) for r in df.itertuples()
            if not valid_npy(outdir/f'{r.prot_id}.npy',(len(r.sequence),512))
        ]

        if not still_missing:
            print(split,': ESM-IF complete -> SKIP')
            continue

        subdf=df[df.prot_id.isin(still_missing)].copy()
        fasta=ESMIF_ROOT/split/f'{split}_missing.fasta'
        fasta.parent.mkdir(parents=True,exist_ok=True)
        write_fasta(subdf,fasta)

        failed=outdir/'failed_ids.txt'
        if failed.exists():
            failed.unlink()

        rr=mex(
            f'python multi_modal_binding/get_esm_if_embedding.py '
            f'"{fasta}" "{outdir}" "{PDB_NORM_ROOT/split}" --chain_id A',
            cwd=ESMB,check=False,capture=True
        )
        print(rr.stdout)
        if rr.returncode!=0:
            raise RuntimeError(f'{split}: ESM-IF subprocess failed')

        if failed.exists():
            bad=[x.strip() for x in failed.read_text().splitlines() if x.strip()]
            if bad:
                raise RuntimeError(f'{split}: ESM-IF failed IDs: {bad[:20]}')

for split,df in SPLIT_DF.items():
    bad=[]
    for r in df.itertuples():
        fp=ESMIF_PATHS[split]/f'{r.prot_id}.npy'
        if not valid_npy(fp,(len(r.sequence),512)):
            bad.append(str(r.prot_id))
    if bad:
        raise RuntimeError(f'{split}: incomplete ESM-IF: {bad[:20]}')
    print(split,f'ESM-IF PASS {len(df)}/{len(df)}')


In [ ]:
for split,df in SPLIT_DF.items():
    store=ctx.stores[SPLIT_STORE_KEY[split]]
    for r in df.itertuples():
        a=np.load(
            ESMIF_PATHS[split]/f'{r.prot_id}.npy'
        ).astype(np.float32,copy=False)

        if a.shape!=(len(r.sequence),512):
            raise RuntimeError(f'{split}/{r.prot_id}: {a.shape}')

        set_block(store,str(r.prot_id),'ESMIF',a)

print('Attached ESMIF.')
display(reporting.dimension_table(ctx.stores['train']))


# Part B — SITECTX32: a cheap probe of the site-hypergraph hypothesis

`SITECTX32` is label-free. It summarizes the higher-order donor neighborhood around each residue:
donor counts, H/C/D/E composition, donor-distance distribution, nearest donor distances,
centroid geometry, sequence separation, and canonical-donor identity.

If SITECTX does not add marginal Dev AUPRC beyond the compact anchor, a full site-hypergraph is unlikely to justify its complexity.
If it does add signal, that is direct evidence to build a residue↔site hypergraph next.


In [ ]:
DONOR_ATOMS={
    'HIS':['ND1','NE2'],
    'CYS':['SG'],
    'ASP':['OD1','OD2'],
    'GLU':['OE1','OE2'],
}
DONOR_TYPES=['HIS','CYS','ASP','GLU']

SITECTX_ROOT=V3_CACHE/'sitectx32_v1'
SITECTX_ROOT.mkdir(parents=True,exist_ok=True)

def atom_mean_coord(res,names):
    coords=[np.asarray(res[n].coord,dtype=np.float32) for n in names if n in res]
    if not coords:
        return None
    return np.mean(np.stack(coords,axis=0),axis=0)

def build_sitectx32(pdb_path,sequence):
    st=PDBParser(QUIET=True).get_structure('p',str(pdb_path))
    model=next(st.get_models())
    chain=model['A']

    residues=[
        r for r in chain
        if r.id[0]==' ' and r.resname.strip().upper() in AA3
    ]

    if len(residues)!=len(sequence):
        raise RuntimeError(f'{pdb_path}: residue length mismatch')

    centers=[]
    donor_centers=[]
    donor_types=[]
    donor_seq_idx=[]

    for i,res in enumerate(residues):
        rn=res.resname.strip().upper()

        if 'CA' in res:
            ca=np.asarray(res['CA'].coord,dtype=np.float32)
        else:
            aa=[np.asarray(a.coord,dtype=np.float32) for a in res]
            if not aa:
                raise RuntimeError(f'{pdb_path}: residue {i} has no atoms')
            ca=np.mean(np.stack(aa),axis=0)

        dcoord=None
        if rn in DONOR_ATOMS:
            dcoord=atom_mean_coord(res,DONOR_ATOMS[rn])

        centers.append(dcoord if dcoord is not None else ca)

        if dcoord is not None:
            donor_centers.append(dcoord)
            donor_types.append(rn)
            donor_seq_idx.append(i)

    centers=np.asarray(centers,dtype=np.float32)

    if donor_centers:
        D=np.asarray(donor_centers,dtype=np.float32)
        donor_seq_idx=np.asarray(donor_seq_idx,dtype=int)
    else:
        D=np.zeros((0,3),dtype=np.float32)
        donor_seq_idx=np.zeros((0,),dtype=int)

    out=np.zeros((len(sequence),32),dtype=np.float32)

    for i,c in enumerate(centers):
        dist=np.linalg.norm(D-c[None,:],axis=1) if len(D) else np.zeros((0,),dtype=np.float32)

        # 1-4: total canonical donor count at 4/6/8/10 A.
        for j,radius in enumerate([4.0,6.0,8.0,10.0]):
            out[i,j]=min(float(np.sum(dist<=radius)),6.0)/6.0

        # 5-12: H/C/D/E composition at 6 and 8 A.
        col=4
        for radius in [6.0,8.0]:
            mask=dist<=radius
            for typ in DONOR_TYPES:
                n=sum(1 for m,t in zip(mask,donor_types) if m and t==typ)
                out[i,col]=min(float(n),6.0)/6.0
                col+=1

        m8=dist<=8.0
        d8=dist[m8]
        idx8=donor_seq_idx[m8] if len(donor_seq_idx) else np.zeros((0,),dtype=int)
        coords8=D[m8] if len(D) else np.zeros((0,3),dtype=np.float32)
        types8=[t for m,t in zip(m8,donor_types) if m]

        # 13-16: donor-distance statistics.
        if len(d8):
            out[i,12:16]=np.clip(
                np.asarray([np.min(d8),np.mean(d8),np.std(d8),np.max(d8)])/10.0,
                0,1
            )

        # 17-20: four nearest donor distances.
        nearest=np.sort(d8)[:4]
        out[i,16:20]=1.0
        if len(nearest):
            out[i,16:16+len(nearest)]=np.clip(nearest/10.0,0,1)

        # 21-24: centroid/group geometry.
        if len(coords8):
            centroid=np.mean(coords8,axis=0)
            to_cent=np.linalg.norm(c-centroid)
            dc=np.linalg.norm(coords8-centroid[None,:],axis=1)

            pair=[]
            if len(coords8)>=2:
                for a in range(len(coords8)):
                    for b in range(a+1,len(coords8)):
                        pair.append(np.linalg.norm(coords8[a]-coords8[b]))
            maxpair=max(pair) if pair else 0.0

            out[i,20:24]=np.clip(
                np.asarray([to_cent,np.mean(dc),np.std(dc),maxpair])/10.0,
                0,1
            )

        # 25-27: H/C/acidic fractions.
        n8=max(len(types8),1)
        out[i,24]=types8.count('HIS')/n8
        out[i,25]=types8.count('CYS')/n8
        out[i,26]=(types8.count('ASP')+types8.count('GLU'))/n8

        # 28: coordination density.
        out[i,27]=min(len(types8),6)/6.0

        # 29-31: sequence separation of nearby donors.
        if len(idx8):
            sep=np.abs(idx8-i)
            out[i,28:31]=np.clip(
                np.asarray([np.min(sep),np.mean(sep),np.max(sep)])/50.0,
                0,1
            )

        # 32: focal residue is canonical H/C/D/E donor.
        out[i,31]=1.0 if sequence[i] in {'H','C','D','E'} else 0.0

    return out

if RUN_SITECTX:
    for split,df in SPLIT_DF.items():
        outdir=SITECTX_ROOT/split
        outdir.mkdir(parents=True,exist_ok=True)

        for n,r in enumerate(df.itertuples(),start=1):
            fp=outdir/f'{r.prot_id}.npy'
            if not FORCE_REBUILD_SITECTX and valid_npy(fp,(len(r.sequence),32)):
                continue

            arr=build_sitectx32(
                PDB_NORM_ROOT/split/f'{r.prot_id}.pdb',
                r.sequence
            )
            np.save(fp,arr)

            if n%100==0:
                print(split,n,'/',len(df))

for split,df in SPLIT_DF.items():
    store=ctx.stores[SPLIT_STORE_KEY[split]]
    for r in df.itertuples():
        fp=SITECTX_ROOT/split/f'{r.prot_id}.npy'
        if not valid_npy(fp,(len(r.sequence),32)):
            raise RuntimeError(f'{split}/{r.prot_id}: SITECTX missing/invalid')
        set_block(store,str(r.prot_id),'SITECTX',np.load(fp).astype(np.float32,copy=False))

print('SITECTX32 attached.')
display(reporting.dimension_table(ctx.stores['train']))


# Part C — Dev-only representation screen with the same standardized MLP

Anchor: `ESM + ATOM + CHEM + MULTI2`.

Only marginal additions are tested:
- `+ESMIF`
- `+SITECTX`
- `+ESMIF+SITECTX`
- `ESM+ESMIF` as a structural complement control

No external result participates in this choice.


In [ ]:
REP_GRID={
    'Anchor_ESM_ATOM_CHEM_MULTI2':
        ANCHOR_BLOCKS,

    'Anchor_plus_ESMIF':
        [*ANCHOR_BLOCKS,'ESMIF'],

    'Anchor_plus_SITECTX':
        [*ANCHOR_BLOCKS,'SITECTX'],

    'Anchor_plus_ESMIF_SITECTX':
        [*ANCHOR_BLOCKS,'ESMIF','SITECTX'],

    'ESM_plus_ESMIF':
        ['ESM','ESMIF'],
}

if RUN_REPRESENTATION_SCREEN:
    rep_cv,rep_folds=suite.run_dev_cv_only(
        ctx,
        REP_GRID,
        models=('MLP',),
        run_version='metal_v3_01_representation_mlp',
    )
    rep_cv.to_csv(RUN_ROOT/'representation_cv.csv',index=False)
    rep_folds.to_csv(RUN_ROOT/'representation_folds.csv',index=False)
else:
    rep_cv=pd.read_csv(RUN_ROOT/'representation_cv.csv')
    rep_folds=pd.read_csv(RUN_ROOT/'representation_folds.csv')

rep_rank=rep_cv.sort_values('CV_AUPRC',ascending=False).reset_index(drop=True)

anchor_score=float(
    rep_cv.loc[
        rep_cv.representation.eq('Anchor_ESM_ATOM_CHEM_MULTI2'),
        'CV_AUPRC'
    ].iloc[0]
)

rep_rank['delta_vs_anchor']=rep_rank['CV_AUPRC']-anchor_score
display(rep_rank)

BEST_REP_NAME=str(rep_rank.iloc[0].representation)
BEST_REP_BLOCKS=list(REP_GRID[BEST_REP_NAME])
BEST_REP_GAIN=float(rep_rank.iloc[0].delta_vs_anchor)

SITE_HYPERGRAPH_SIGNAL=bool(
    any(
        (rep_rank.representation.eq(name) & (rep_rank.delta_vs_anchor>=MIN_MEAN_AUPRC_GAIN)).any()
        for name in ['Anchor_plus_SITECTX','Anchor_plus_ESMIF_SITECTX']
    )
)

print('Anchor CV AUPRC:',anchor_score)
print('Best representation:',BEST_REP_NAME)
print('Gain vs anchor:',BEST_REP_GAIN)
print('Locked blocks:',BEST_REP_BLOCKS)
print('SITE_HYPERGRAPH_SIGNAL:',SITE_HYPERGRAPH_SIGNAL)

(RUN_ROOT/'representation_decision.json').write_text(
    json.dumps({
        'anchor_cv_auprc':anchor_score,
        'best_representation':BEST_REP_NAME,
        'best_blocks':BEST_REP_BLOCKS,
        'gain_vs_anchor':BEST_REP_GAIN,
        'site_hypergraph_signal':SITE_HYPERGRAPH_SIGNAL,
    },indent=2)
)


# Part D — Multimodal architecture screen

The feature set is now frozen from Dev.

Modalities:
- `ESM`
- `ESMIF` if selected
- `EXPLICIT` = all remaining selected blocks

Architectures:
1. ConcatMLP
2. ProjectedConcat
3. GatedFusion
4. ModalityAttention
5. LatentMetalMoE

All use the same official five protein folds. The external sets stay untouched.


In [ ]:
def modality_spec(blocks):
    out={}
    if 'ESM' in blocks:
        out['ESM']=['ESM']
    if 'ESMIF' in blocks:
        out['ESMIF']=['ESMIF']
    explicit=[b for b in blocks if b not in {'ESM','ESMIF'}]
    if explicit:
        out['EXPLICIT']=explicit
    return out

MODALITY_SPEC=modality_spec(BEST_REP_BLOCKS)
print('Modalities:',MODALITY_SPEC)

def concat_blocks(store,pid,blocks):
    arr=[get_block(store,pid,b).astype(np.float32,copy=False) for b in blocks]
    lens={a.shape[0] for a in arr}
    if len(lens)!=1:
        raise RuntimeError(f'{pid}: block length mismatch')
    return np.concatenate(arr,axis=1)

def split_pack(split):
    df=SPLIT_DF[split]
    store=ctx.stores[SPLIT_STORE_KEY[split]]

    mods={k:[] for k in MODALITY_SPEC}
    labels=[]
    pids=[]
    ridx=[]

    for r in df.itertuples():
        L=len(r.sequence)
        for m,blocks in MODALITY_SPEC.items():
            mods[m].append(concat_blocks(store,str(r.prot_id),blocks))
        labels.append(np.asarray(r.metal_binding,dtype=np.float32))
        pids.extend([str(r.prot_id)]*L)
        ridx.extend(range(L))

    mods={
        k:np.concatenate(v,axis=0).astype(
            np.float16 if k in {'ESM','ESMIF'} else np.float32,
            copy=False
        )
        for k,v in mods.items()
    }

    return {
        'mods':mods,
        'y':np.concatenate(labels).astype(np.float32),
        'pids':np.asarray(pids,dtype=object),
        'ridx':np.asarray(ridx,dtype=np.int32),
    }

PACKS={split:split_pack(split) for split in SPLIT_DF}

for split,p in PACKS.items():
    print(
        split,
        {k:v.shape for k,v in p['mods'].items()},
        'labels',p['y'].shape,
        'positives',int(p['y'].sum())
    )


In [ ]:
torch.manual_seed(CUSTOM_SEED)
np.random.seed(CUSTOM_SEED)

class ModalityLayerNorm(nn.Module):
    def __init__(self,dims):
        super().__init__()
        self.norms=nn.ModuleDict({k:nn.LayerNorm(v) for k,v in dims.items()})
    def forward(self,x):
        return {k:self.norms[k](v) for k,v in x.items()}

class ConcatMLP(nn.Module):
    def __init__(self,dims,drop=0.2):
        super().__init__()
        self.norm=ModalityLayerNorm(dims)
        d=sum(dims.values())
        self.head=nn.Sequential(
            nn.Linear(d,512),nn.GELU(),nn.Dropout(drop),
            nn.Linear(512,128),nn.GELU(),nn.Dropout(drop),
            nn.Linear(128,1),
        )
    def forward(self,x):
        z=self.norm(x)
        return self.head(torch.cat([z[k] for k in sorted(z)],dim=-1)).squeeze(-1)

class ProjectedConcat(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()
        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),nn.Linear(d,latent),nn.GELU(),nn.Dropout(drop)
            ) for k,d in dims.items()
        })
        self.head=nn.Sequential(
            nn.Linear(latent*len(dims),256),nn.GELU(),nn.Dropout(drop),
            nn.Linear(256,1),
        )
    def forward(self,x):
        z=[self.enc[k](x[k]) for k in sorted(self.enc)]
        return self.head(torch.cat(z,dim=-1)).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2,noise=0.03):
        super().__init__()
        self.noise=noise
        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),nn.Linear(d,latent),nn.GELU(),nn.Dropout(drop)
            ) for k,d in dims.items()
        })
        self.gate=nn.ModuleDict({k:nn.Linear(latent,1) for k in dims})
        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),nn.GELU(),nn.Dropout(drop),
            nn.Linear(192,1),
        )
    def forward(self,x):
        zz=[]
        gg=[]
        for k in sorted(self.enc):
            v=x[k]
            if self.training and self.noise>0:
                v=v+self.noise*torch.randn_like(v)
            z=self.enc[k](v)
            zz.append(z)
            gg.append(self.gate[k](z))
        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)
        fused=(Z*G).sum(dim=1)
        return self.head(fused).squeeze(-1)

class ModalityAttention(nn.Module):
    def __init__(self,dims,latent=192,heads=6,drop=0.2):
        super().__init__()
        assert latent%heads==0
        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),nn.Linear(d,latent),nn.GELU(),nn.Dropout(drop)
            ) for k,d in dims.items()
        })
        self.attn=nn.MultiheadAttention(
            embed_dim=latent,num_heads=heads,dropout=drop,batch_first=True
        )
        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),nn.GELU(),nn.Dropout(drop),
            nn.Linear(192,1),
        )
    def forward(self,x):
        Z=torch.stack([self.enc[k](x[k]) for k in sorted(self.enc)],dim=1)
        A,_=self.attn(Z,Z,Z,need_weights=False)
        fused=(Z+A).mean(dim=1)
        return self.head(fused).squeeze(-1)

class LatentMetalMoE(nn.Module):
    def __init__(self,dims,latent=192,experts=4,drop=0.2):
        super().__init__()
        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),nn.Linear(d,latent),nn.GELU(),nn.Dropout(drop)
            ) for k,d in dims.items()
        })
        self.mod_gate=nn.ModuleDict({k:nn.Linear(latent,1) for k in dims})
        self.shared=nn.Sequential(
            nn.LayerNorm(latent),nn.Linear(latent,latent),nn.GELU()
        )
        self.experts=nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent,128),nn.GELU(),nn.Dropout(drop),nn.Linear(128,1)
            ) for _ in range(experts)
        ])
        self.expert_gate=nn.Linear(latent,experts)

    def forward(self,x):
        zz=[]
        gg=[]
        for k in sorted(self.enc):
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.mod_gate[k](z))
        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)
        h=self.shared((Z*G).sum(dim=1))
        expert_logits=torch.cat([e(h) for e in self.experts],dim=1)
        weights=torch.softmax(self.expert_gate(h),dim=1)
        return (expert_logits*weights).sum(dim=1)

ARCH_FACTORY={
    'ConcatMLP':ConcatMLP,
    'ProjectedConcat':ProjectedConcat,
    'GatedFusion':GatedFusion,
    'ModalityAttention':ModalityAttention,
    'LatentMetalMoE':LatentMetalMoE,
}

DIMS={k:v.shape[1] for k,v in PACKS['DevSet1010']['mods'].items()}
print('Modality dimensions:',DIMS)


In [ ]:
class MultiArrayDataset(Dataset):
    def __init__(self,mods,y):
        self.mods=mods
        self.y=y
        self.keys=sorted(mods)
    def __len__(self):
        return len(self.y)
    def __getitem__(self,i):
        x={
            k:torch.from_numpy(np.asarray(self.mods[k][i],dtype=np.float32))
            for k in self.keys
        }
        return x,torch.tensor(self.y[i],dtype=torch.float32)

def collate_multi(batch):
    keys=sorted(batch[0][0])
    x={k:torch.stack([b[0][k] for b in batch],dim=0) for k in keys}
    y=torch.stack([b[1] for b in batch],dim=0)
    return x,y

def ap(y,p):
    return float(average_precision_score(y,p))

def predict_model(model,dataset,indices=None,batch_size=8192):
    ds=dataset if indices is None else Subset(dataset,indices.tolist())
    loader=DataLoader(
        ds,batch_size=batch_size,shuffle=False,num_workers=0,
        pin_memory=True,collate_fn=collate_multi
    )
    model.eval()
    out=[]
    with torch.no_grad():
        for x,_ in loader:
            x={k:v.to(DEVICE,non_blocking=True) for k,v in x.items()}
            out.append(torch.sigmoid(model(x)).cpu().numpy())
    return np.concatenate(out)

def train_one_fold(arch_name,pack,train_idx,val_idx,fold):
    torch.manual_seed(CUSTOM_SEED+fold)
    np.random.seed(CUSTOM_SEED+fold)

    dataset=MultiArrayDataset(pack['mods'],pack['y'])
    train_loader=DataLoader(
        Subset(dataset,train_idx.tolist()),
        batch_size=CUSTOM_BATCH,shuffle=True,num_workers=0,
        pin_memory=True,collate_fn=collate_multi
    )
    val_loader=DataLoader(
        Subset(dataset,val_idx.tolist()),
        batch_size=8192,shuffle=False,num_workers=0,
        pin_memory=True,collate_fn=collate_multi
    )

    model=ARCH_FACTORY[arch_name](DIMS).to(DEVICE)

    ytr=pack['y'][train_idx]
    npos=max(float(ytr.sum()),1.0)
    nneg=max(float(len(ytr)-ytr.sum()),1.0)
    pos_weight=float(np.sqrt(nneg/npos))

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight,device=DEVICE)
    )
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=CUSTOM_LR,
        weight_decay=CUSTOM_WEIGHT_DECAY
    )

    best_ap=-1.0
    best_state=None
    bad_epochs=0

    for epoch in range(1,CUSTOM_MAX_EPOCHS+1):
        model.train()
        losses=[]

        for x,y in train_loader:
            x={k:v.to(DEVICE,non_blocking=True) for k,v in x.items()}
            y=y.to(DEVICE,non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits=model(x)
            loss=criterion(logits,y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step()
            losses.append(float(loss.detach().cpu()))

        model.eval()
        vp=[]
        vy=[]
        with torch.no_grad():
            for x,y in val_loader:
                x={k:v.to(DEVICE,non_blocking=True) for k,v in x.items()}
                vp.append(torch.sigmoid(model(x)).cpu().numpy())
                vy.append(y.numpy())

        vp=np.concatenate(vp)
        vy=np.concatenate(vy)
        val_ap=ap(vy,vp)

        if val_ap>best_ap+1e-5:
            best_ap=val_ap
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad_epochs=0
        else:
            bad_epochs+=1

        if epoch==1 or epoch%5==0:
            print(
                arch_name,'fold',fold,'epoch',epoch,
                'loss',round(float(np.mean(losses)),4),
                'val_AP',round(val_ap,5),
                'best',round(best_ap,5)
            )

        if bad_epochs>=CUSTOM_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model,best_ap,best_state

def run_arch_cv(arch_name):
    pack=PACKS['DevSet1010']
    pids=pack['pids']
    y=pack['y']
    oof=np.full(len(y),np.nan,dtype=np.float32)

    fold_rows=[]
    states={}

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(val_ids)
        is_val=np.fromiter((p in val_set for p in pids),dtype=bool,count=len(pids))
        val_idx=np.flatnonzero(is_val)
        train_idx=np.flatnonzero(~is_val)

        model,best_ap,state=train_one_fold(
            arch_name,pack,train_idx,val_idx,fold
        )

        pred=predict_model(
            model,
            MultiArrayDataset(pack['mods'],pack['y']),
            indices=val_idx
        )
        oof[val_idx]=pred
        states[fold]=state

        fold_rows.append({
            'architecture':arch_name,
            'fold':fold,
            'AUPRC':ap(y[val_idx],pred),
            'n_val_residues':len(val_idx),
            'n_val_positive':int(y[val_idx].sum()),
        })

        del model
        torch.cuda.empty_cache()

    if not np.all(np.isfinite(oof)):
        raise RuntimeError(f'{arch_name}: incomplete OOF predictions')

    fold_df=pd.DataFrame(fold_rows)
    summary={
        'architecture':arch_name,
        'CV_AUPRC_mean':float(fold_df.AUPRC.mean()),
        'CV_AUPRC_std':float(fold_df.AUPRC.std(ddof=1)),
        'OOF_AUPRC':ap(y,oof),
    }

    return summary,fold_df,oof,states


## Cache-safe custom architecture CV

V4 persists each completed architecture/fold state and its validation prediction.  
If Colab disconnects, completed folds are reused instead of retrained.


In [ ]:
CUSTOM_STATE_ROOT = RUN_ROOT/'custom_fusion_states'
CUSTOM_STATE_ROOT.mkdir(parents=True, exist_ok=True)

def run_arch_cv_cached(arch_name):
    pack = PACKS['DevSet1010']
    pids = pack['pids']
    y = pack['y']
    dataset = MultiArrayDataset(pack['mods'], pack['y'])
    oof = np.full(len(y), np.nan, dtype=np.float32)

    fold_rows = []
    states = {}

    for fold, val_ids in OFFICIAL_FOLDS.items():
        val_set = set(val_ids)
        val_idx = np.flatnonzero(np.array([p in val_set for p in pids]))
        train_idx = np.flatnonzero(np.array([p not in val_set for p in pids]))

        adir = CUSTOM_STATE_ROOT/arch_name
        adir.mkdir(parents=True, exist_ok=True)

        state_fp = adir/f'fold{fold}.pt'
        pred_fp = adir/f'fold{fold}_val_pred.npy'
        meta_fp = adir/f'fold{fold}_meta.json'

        reuse = False
        if state_fp.exists() and pred_fp.exists() and meta_fp.exists():
            try:
                meta = json.loads(meta_fp.read_text())
                vp = np.load(pred_fp)
                reuse = (
                    meta.get('architecture') == arch_name
                    and meta.get('representation') == BEST_REP_NAME
                    and meta.get('n_val') == len(val_idx)
                    and vp.shape == (len(val_idx),)
                    and np.all(np.isfinite(vp))
                )
            except Exception:
                reuse = False

        if reuse:
            print(f'{arch_name} fold {fold}: REUSE')
            state = torch.load(state_fp, map_location='cpu')
            pred = np.load(pred_fp).astype(np.float32)
            states[fold] = state
        else:
            print(f'{arch_name} fold {fold}: TRAIN')
            model, _, state = train_one_fold(
                arch_name, pack, train_idx, val_idx, fold
            )
            pred = predict_model(
                model, dataset, indices=val_idx
            ).astype(np.float32)

            torch.save(state, state_fp)
            np.save(pred_fp, pred)
            meta_fp.write_text(json.dumps({
                'architecture': arch_name,
                'representation': BEST_REP_NAME,
                'blocks': BEST_REP_BLOCKS,
                'n_val': int(len(val_idx)),
            }, indent=2))

            states[fold] = state
            del model
            torch.cuda.empty_cache()

        oof[val_idx] = pred

        fold_rows.append({
            'architecture': arch_name,
            'fold': fold,
            'AUPRC': ap(y[val_idx], pred),
            'n_val_residues': len(val_idx),
            'n_val_positive': int(y[val_idx].sum()),
        })

    if not np.all(np.isfinite(oof)):
        raise RuntimeError(f'{arch_name}: incomplete OOF predictions')

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        'architecture': arch_name,
        'CV_AUPRC_mean': float(fold_df.AUPRC.mean()),
        'CV_AUPRC_std': float(fold_df.AUPRC.std(ddof=1)),
        'OOF_AUPRC': ap(y, oof),
    }
    return summary, fold_df, oof, states


In [ ]:
ARCH_RESULTS={}
ARCH_STATES={}
ARCH_OOF={}
fold_tables=[]

if RUN_FUSION_SCREEN:
    for arch in ARCH_FACTORY:
        print('\n'+'='*72)
        print('ARCHITECTURE:',arch)
        print('='*72)

        summary,fold_df,oof,states=run_arch_cv_cached(arch)
        ARCH_RESULTS[arch]=summary
        ARCH_STATES[arch]=states
        ARCH_OOF[arch]=oof
        fold_tables.append(fold_df)

        pd.DataFrame({
            'prot_id':PACKS['DevSet1010']['pids'],
            'residue_index':PACKS['DevSet1010']['ridx'],
            'y_true':PACKS['DevSet1010']['y'].astype(int),
            'probability':oof,
        }).to_csv(RUN_ROOT/f'OOF_{arch}.csv.gz',index=False)

arch_summary=pd.DataFrame(ARCH_RESULTS.values()).sort_values(
    'CV_AUPRC_mean',ascending=False
).reset_index(drop=True)
arch_folds=pd.concat(fold_tables,ignore_index=True)

arch_summary.to_csv(RUN_ROOT/'fusion_architecture_cv.csv',index=False)
arch_folds.to_csv(RUN_ROOT/'fusion_architecture_folds.csv',index=False)

display(arch_summary)

concat_score=float(
    arch_summary.loc[
        arch_summary.architecture.eq('ConcatMLP'),
        'CV_AUPRC_mean'
    ].iloc[0]
)

best_arch=str(arch_summary.iloc[0].architecture)
best_arch_score=float(arch_summary.iloc[0].CV_AUPRC_mean)
gain_arch=best_arch_score-concat_score

LOCKED_ARCH='ConcatMLP' if gain_arch<MIN_MEAN_AUPRC_GAIN else best_arch

print('Concat baseline:',concat_score)
print('Best architecture:',best_arch,best_arch_score)
print('Architecture gain:',gain_arch)
print('LOCKED_ARCH:',LOCKED_ARCH)

(RUN_ROOT/'fusion_decision.json').write_text(
    json.dumps({
        'best_representation':BEST_REP_NAME,
        'best_blocks':BEST_REP_BLOCKS,
        'concat_cv_auprc':concat_score,
        'best_architecture':best_arch,
        'best_architecture_cv_auprc':best_arch_score,
        'gain_vs_concat':gain_arch,
        'locked_architecture':LOCKED_ARCH,
    },indent=2)
)


# Part E — Existing GEAR residual graph on the locked representation

This is the direct graph test.

The graph receives exactly the representation selected above and uses:
- rich relation edges;
- donor-focused relation edges.

The decision is made from Dev CV only.
The graph backend may internally compute external outputs, but they are not used for selection.


In [ ]:
GRAPH_EXPERIMENTS={
    'ResidualGraph_Locked_Rich':{
        'blocks':BEST_REP_BLOCKS,
        'architecture':'residual',
        'edge_mode':'rich',
    },
    'ResidualGraph_Locked_Donor':{
        'blocks':BEST_REP_BLOCKS,
        'architecture':'residual',
        'edge_mode':'donor',
    },
}

graph_cv=None
graph_ext=None

if RUN_GRAPH_SCREEN:
    graph_cv,graph_ext=graph.run_graph_grid(
        ctx.stores,
        ctx.fold_ids,
        BASE,
        GRAPH_EXPERIMENTS,
        run_version='metal_v3_03_locked_residual_graph',
        cfg=graph.GraphConfig(),
        device=DEVICE,
    )

    graph_cv.to_csv(RUN_ROOT/'graph_cv.csv',index=False)
    graph_ext.to_csv(RUN_ROOT/'graph_external_hidden_until_lock.csv',index=False)

    print('DEV-ONLY GRAPH RESULTS')
    display(graph_cv.sort_values('CV_AUPRC',ascending=False))

    best_graph=float(graph_cv.CV_AUPRC.max())
    graph_gain_vs_suite_anchor=best_graph-anchor_score

    print('Best graph CV AUPRC:',best_graph)
    print('Gain vs standardized anchor MLP:',graph_gain_vs_suite_anchor)

    graph_promising=bool(graph_gain_vs_suite_anchor>=MIN_MEAN_AUPRC_GAIN)

    (RUN_ROOT/'graph_decision.json').write_text(
        json.dumps({
            'best_graph_cv_auprc':best_graph,
            'gain_vs_suite_anchor':graph_gain_vs_suite_anchor,
            'promising':graph_promising,
        },indent=2)
    )
else:
    print('Graph screen OFF.')


# Full candidate metal-site hypergraph

V4 now builds an actual trainable bipartite hypergraph.

Candidate sites are generated without labels from spatially compatible His/Cys/Asp/Glu donor neighborhoods.

Residue embeddings are aggregated into candidate-site nodes, transformed, then sent back as a small residual correction to the residue representation.


In [ ]:
HG_CACHE_ROOT = V3_CACHE/'candidate_site_hypergraph_v1'
HG_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

HG_DONOR_ATOMS = {
    'HIS': ['ND1','NE2'],
    'CYS': ['SG'],
    'ASP': ['OD1','OD2'],
    'GLU': ['OE1','OE2'],
}
HG_DONOR_ORDER = ['HIS','CYS','ASP','GLU']

def _donor_coord(res):
    rn = res.resname.strip().upper()
    names = HG_DONOR_ATOMS.get(rn)
    if names is None:
        return None
    coords = [
        np.asarray(res[n].coord, dtype=np.float32)
        for n in names if n in res
    ]
    if not coords:
        return None
    return np.mean(np.stack(coords, axis=0), axis=0)

def _candidate_sites_from_pdb(pdb_path, sequence):
    st = PDBParser(QUIET=True).get_structure('hg', str(pdb_path))
    model = next(st.get_models())
    chain = model['A']

    residues = [
        r for r in chain
        if r.id[0] == ' ' and r.resname.strip().upper() in AA3
    ]
    if len(residues) != len(sequence):
        raise RuntimeError(
            f'{pdb_path}: residue count {len(residues)} != sequence {len(sequence)}'
        )

    donor_idx, donor_type, donor_coord = [], [], []

    for i, res in enumerate(residues):
        c = _donor_coord(res)
        if c is not None:
            donor_idx.append(i)
            donor_type.append(res.resname.strip().upper())
            donor_coord.append(c)

    if len(donor_idx) < 2:
        return (
            np.zeros((0,), dtype=np.int64),
            np.zeros((0,), dtype=np.int64),
            np.zeros((0,24), dtype=np.float32),
        )

    donor_idx = np.asarray(donor_idx, dtype=np.int64)
    donor_coord = np.asarray(donor_coord, dtype=np.float32)

    dd = np.linalg.norm(
        donor_coord[:,None,:] - donor_coord[None,:,:],
        axis=-1
    )

    signatures = set()

    for i in range(len(donor_idx)):
        nei = np.flatnonzero(dd[i] <= HG_RADIUS_A)
        nei = nei[np.argsort(dd[i,nei])]
        nei = nei[:HG_MAX_DONORS]
        if len(nei) >= 2:
            signatures.add(tuple(sorted(donor_idx[nei].tolist())))

    for i in range(len(donor_idx)):
        for j in range(i+1, len(donor_idx)):
            if dd[i,j] <= 5.0:
                signatures.add(tuple(sorted([
                    int(donor_idx[i]),
                    int(donor_idx[j])
                ])))

    signatures = sorted(signatures)

    idx_to_meta = {
        int(ridx): (coord, typ)
        for ridx, coord, typ in zip(donor_idx, donor_coord, donor_type)
    }

    edge_res, edge_site, site_feat = [], [], []

    for sid, sig in enumerate(signatures):
        coords = np.stack(
            [idx_to_meta[int(r)][0] for r in sig],
            axis=0
        )
        types = [idx_to_meta[int(r)][1] for r in sig]
        n = len(sig)

        pair, seq_pair = [], []
        for a in range(n):
            for b in range(a+1,n):
                pair.append(float(np.linalg.norm(coords[a]-coords[b])))
                seq_pair.append(abs(int(sig[a])-int(sig[b])))

        pair = np.asarray(pair if pair else [0.0], dtype=np.float32)
        seq_pair = np.asarray(seq_pair if seq_pair else [0.0], dtype=np.float32)

        centroid = coords.mean(axis=0)
        radial = np.linalg.norm(coords-centroid[None,:], axis=1)

        if n >= 2:
            centered = coords-centroid[None,:]
            cov = centered.T @ centered / max(n-1,1)
            eig = np.sort(np.clip(np.linalg.eigvalsh(cov),0,None))[::-1]
        else:
            eig = np.zeros(3, dtype=np.float32)

        counts = np.array(
            [types.count(t) for t in HG_DONOR_ORDER],
            dtype=np.float32
        )
        acidic = counts[2] + counts[3]

        f = np.zeros(24, dtype=np.float32)
        f[0] = min(n, HG_MAX_DONORS) / HG_MAX_DONORS
        f[1:5] = np.clip(counts/6.0, 0, 1)
        f[5] = counts[0]/max(n,1)
        f[6] = counts[1]/max(n,1)
        f[7] = acidic/max(n,1)

        f[8:12] = np.clip(
            np.array([pair.min(), pair.mean(), pair.std(), pair.max()])/10.0,
            0, 1
        )
        f[12:15] = np.clip(
            np.array([radial.mean(), radial.std(), radial.max()])/8.0,
            0, 1
        )
        f[15:18] = np.clip(
            np.array([seq_pair.min(), seq_pair.mean(), seq_pair.max()])/50.0,
            0, 1
        )
        f[18:21] = np.clip(eig/25.0, 0, 1)
        f[21] = min(max(sig)-min(sig), 100)/100.0
        f[22] = float(len(set(types)) >= 2)
        f[23] = float(n >= 3)

        site_feat.append(f)

        for r in sig:
            edge_res.append(int(r))
            edge_site.append(int(sid))

    return (
        np.asarray(edge_res, dtype=np.int64),
        np.asarray(edge_site, dtype=np.int64),
        np.asarray(site_feat, dtype=np.float32).reshape(-1,24),
    )

HG_RAM = {}
HG_AUDIT = []

for split, df in SPLIT_DF.items():
    outdir = HG_CACHE_ROOT/split
    outdir.mkdir(parents=True, exist_ok=True)

    for n, r in enumerate(df.itertuples(), start=1):
        pid = str(r.prot_id)
        fp = outdir/f'{pid}.npz'

        ok = False
        if fp.exists():
            try:
                z = np.load(fp)
                ok = (
                    z['site_feat'].ndim == 2
                    and z['site_feat'].shape[1] == 24
                    and len(z['edge_res']) == len(z['edge_site'])
                )
            except Exception:
                ok = False

        if not ok:
            er, es, sf = _candidate_sites_from_pdb(
                PDB_NORM_ROOT/split/f'{pid}.pdb',
                r.sequence
            )
            np.savez_compressed(fp, edge_res=er, edge_site=es, site_feat=sf)

        z = np.load(fp)
        er = z['edge_res'].astype(np.int64)
        es = z['edge_site'].astype(np.int64)
        sf = z['site_feat'].astype(np.float32)

        HG_RAM[(split,pid)] = (er,es,sf)

        covered = len(np.unique(er)) if len(er) else 0
        HG_AUDIT.append({
            'dataset': split,
            'prot_id': pid,
            'n_residues': len(r.sequence),
            'n_candidate_sites': int(len(sf)),
            'n_memberships': int(len(er)),
            'n_site_covered_residues': int(covered),
            'site_covered_fraction': covered/max(len(r.sequence),1),
        })

        if n % 200 == 0:
            print(split, n, '/', len(df))

hg_audit = pd.DataFrame(HG_AUDIT)
hg_audit.to_csv(RUN_ROOT/'hypergraph_site_audit.csv', index=False)

display(
    hg_audit.groupby('dataset').agg(
        proteins=('prot_id','nunique'),
        mean_sites=('n_candidate_sites','mean'),
        median_sites=('n_candidate_sites','median'),
        mean_memberships=('n_memberships','mean'),
        mean_site_coverage=('site_covered_fraction','mean'),
    ).reset_index()
)

print('Candidate-site hypergraph cache: PASS')


In [ ]:
class HGProteinDataset(Dataset):
    def __init__(self, split, df):
        self.split = split
        self.df = df.reset_index(drop=True)
        self.store = ctx.stores[SPLIT_STORE_KEY[split]]
        self.modality_spec = MODALITY_SPEC

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        pid = str(r.prot_id)

        mods = {
            m: concat_blocks(self.store, pid, blocks).astype(np.float32, copy=False)
            for m, blocks in self.modality_spec.items()
        }
        y = np.asarray(r.metal_binding, dtype=np.float32)
        er, es, sf = HG_RAM[(self.split,pid)]

        return {
            'pid': pid,
            'mods': mods,
            'y': y,
            'edge_res': er,
            'edge_site': es,
            'site_feat': sf,
        }

def collate_hg(batch):
    keys = sorted(batch[0]['mods'])

    mods = {
        k: torch.from_numpy(
            np.concatenate([b['mods'][k] for b in batch], axis=0)
        ).float()
        for k in keys
    }

    y = torch.from_numpy(
        np.concatenate([b['y'] for b in batch], axis=0)
    ).float()

    site_feats, edge_res, edge_site = [], [], []
    res_offset = 0
    site_offset = 0
    lengths, pids = [], []

    for b in batch:
        L = len(b['y'])
        S = len(b['site_feat'])
        lengths.append(L)
        pids.append(b['pid'])

        if S:
            site_feats.append(torch.from_numpy(b['site_feat']).float())

        if len(b['edge_res']):
            edge_res.append(torch.from_numpy(b['edge_res']).long() + res_offset)
            edge_site.append(torch.from_numpy(b['edge_site']).long() + site_offset)

        res_offset += L
        site_offset += S

    site_feat = (
        torch.cat(site_feats, dim=0)
        if site_feats
        else torch.zeros((0,24), dtype=torch.float32)
    )

    edge_res = (
        torch.cat(edge_res, dim=0)
        if edge_res
        else torch.zeros((0,), dtype=torch.long)
    )
    edge_site = (
        torch.cat(edge_site, dim=0)
        if edge_site
        else torch.zeros((0,), dtype=torch.long)
    )

    return {
        'pids': pids,
        'lengths': lengths,
        'mods': mods,
        'y': y,
        'edge_res': edge_res,
        'edge_site': edge_site,
        'site_feat': site_feat,
    }

print('HG dataset adapter: PASS')


In [ ]:
class HGResidueEncoder(nn.Module):
    def __init__(self, dims, mode='gated', latent=192, drop=0.2):
        super().__init__()
        self.mode = mode
        self.keys = sorted(dims)

        self.enc = nn.ModuleDict({
            k: nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        if mode == 'gated':
            self.gate = nn.ModuleDict({
                k: nn.Linear(latent,1) for k in dims
            })
        elif mode == 'attention':
            heads = 6
            assert latent % heads == 0
            self.attn = nn.MultiheadAttention(
                latent, heads, dropout=drop, batch_first=True
            )
        else:
            raise ValueError(mode)

    def forward(self, x):
        Z = torch.stack(
            [self.enc[k](x[k]) for k in self.keys],
            dim=1
        )

        if self.mode == 'gated':
            G = torch.cat(
                [self.gate[k](Z[:,j,:]) for j,k in enumerate(self.keys)],
                dim=1
            )
            G = torch.softmax(G, dim=1).unsqueeze(-1)
            h = (Z*G).sum(dim=1)
            aux = {'modality_gate': G.squeeze(-1)}
        else:
            A, attn = self.attn(
                Z, Z, Z,
                need_weights=True,
                average_attn_weights=True
            )
            h = (Z+A).mean(dim=1)
            aux = {'modality_attention': attn}

        return h, aux

class HGResidualNet(nn.Module):
    def __init__(
        self, dims, encoder_mode='gated',
        latent=192, drop=0.2,
        moe=False, experts=4,
    ):
        super().__init__()
        self.encoder_mode = encoder_mode
        self.moe = moe

        self.encoder = HGResidueEncoder(
            dims, mode=encoder_mode, latent=latent, drop=drop
        )

        self.site_feat_proj = nn.Sequential(
            nn.LayerNorm(24),
            nn.Linear(24,64),
            nn.GELU()
        )

        self.site_mlp = nn.Sequential(
            nn.Linear(latent+64, latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent, latent),
            nn.GELU(),
        )

        self.residual_mlp = nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent, latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent, latent),
        )

        self.alpha_logit = nn.Parameter(
            torch.tensor(-2.944, dtype=torch.float32)
        )

        if not moe:
            self.head = nn.Sequential(
                nn.LayerNorm(latent),
                nn.Linear(latent,128),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(128,1)
            )
        else:
            self.experts = nn.ModuleList([
                nn.Sequential(
                    nn.LayerNorm(latent),
                    nn.Linear(latent,128),
                    nn.GELU(),
                    nn.Dropout(drop),
                    nn.Linear(128,1)
                )
                for _ in range(experts)
            ])
            self.expert_gate = nn.Linear(latent,experts)

    def _index_mean(self, values, index, n):
        if n == 0:
            return values.new_zeros((0, values.shape[-1]))

        out = values.new_zeros((n, values.shape[-1]))
        cnt = values.new_zeros((n,1))

        if len(index):
            out.index_add_(0,index,values)
            cnt.index_add_(
                0, index,
                values.new_ones((len(index),1))
            )

        return out / cnt.clamp_min(1.0)

    def forward(
        self, mods, edge_res, edge_site, site_feat,
        return_aux=False
    ):
        h, aux = self.encoder(mods)
        n_res = h.shape[0]
        n_site = site_feat.shape[0]
        alpha = torch.sigmoid(self.alpha_logit)

        if n_site > 0 and edge_res.numel() > 0:
            site_mean = self._index_mean(
                h[edge_res], edge_site, n_site
            )
            sf = self.site_feat_proj(site_feat)
            hs = self.site_mlp(
                torch.cat([site_mean,sf],dim=-1)
            )

            res_site = self._index_mean(
                hs[edge_site], edge_res, n_res
            )

            incident = h.new_zeros((n_res,1))
            incident.index_add_(
                0, edge_res,
                h.new_ones((len(edge_res),1))
            )

            delta = self.residual_mlp(res_site)
            delta = delta * (incident > 0).float()
            h_final = h + alpha*delta
        else:
            h_final = h

        if not self.moe:
            logits = self.head(h_final).squeeze(-1)
        else:
            expert_logits = torch.cat(
                [e(h_final) for e in self.experts],
                dim=1
            )
            expert_weights = torch.softmax(
                self.expert_gate(h_final),
                dim=1
            )
            logits = (expert_logits*expert_weights).sum(dim=1)
            aux['expert_weights'] = expert_weights

        aux['hypergraph_alpha'] = alpha
        return (logits, aux) if return_aux else logits

HG_FACTORY = {
    'HG_GatedResidual': lambda dims: HGResidualNet(
        dims, encoder_mode='gated', latent=HG_LATENT, moe=False
    ),
    'HG_AttentionResidual': lambda dims: HGResidualNet(
        dims, encoder_mode='attention', latent=HG_LATENT, moe=False
    ),
    'HG_GatedMoE': lambda dims: HGResidualNet(
        dims, encoder_mode='gated', latent=HG_LATENT, moe=True, experts=4
    ),
}

print('Hypergraph architectures:', list(HG_FACTORY))


In [ ]:
HG_STATE_ROOT = RUN_ROOT/'hypergraph_states'
HG_STATE_ROOT.mkdir(parents=True, exist_ok=True)

def move_hg_batch(b):
    return {
        **b,
        'mods': {k:v.to(DEVICE, non_blocking=True) for k,v in b['mods'].items()},
        'y': b['y'].to(DEVICE, non_blocking=True),
        'edge_res': b['edge_res'].to(DEVICE, non_blocking=True),
        'edge_site': b['edge_site'].to(DEVICE, non_blocking=True),
        'site_feat': b['site_feat'].to(DEVICE, non_blocking=True),
    }

def hg_predict(model, dataset):
    loader = DataLoader(
        dataset,
        batch_size=HG_PROTEIN_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_hg
    )

    pred, y = [], []

    model.eval()
    with torch.no_grad():
        for b in loader:
            b = move_hg_batch(b)
            p = torch.sigmoid(
                model(
                    b['mods'],
                    b['edge_res'],
                    b['edge_site'],
                    b['site_feat'],
                )
            )
            pred.append(p.detach().cpu().numpy())
            y.append(b['y'].detach().cpu().numpy())

    return (
        np.concatenate(pred).astype(np.float32),
        np.concatenate(y).astype(np.float32),
    )

def hg_train_fold(arch_name, train_ds, val_ds, fold):
    torch.manual_seed(CUSTOM_SEED + 100 + fold)
    np.random.seed(CUSTOM_SEED + 100 + fold)

    model = HG_FACTORY[arch_name](DIMS).to(DEVICE)

    train_y = np.concatenate([
        np.asarray(x, dtype=np.float32)
        for x in train_ds.df.metal_binding
    ])

    npos = max(float(train_y.sum()),1.0)
    nneg = max(float(len(train_y)-train_y.sum()),1.0)
    pos_weight = float(np.sqrt(nneg/npos))

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight,device=DEVICE)
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=HG_LR,
        weight_decay=HG_WEIGHT_DECAY
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=HG_PROTEIN_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_hg
    )

    best_ap = -1.0
    best_state = None
    bad = 0

    for epoch in range(1, HG_MAX_EPOCHS+1):
        model.train()
        losses = []

        for b in train_loader:
            b = move_hg_batch(b)

            opt.zero_grad(set_to_none=True)

            logits = model(
                b['mods'],
                b['edge_res'],
                b['edge_site'],
                b['site_feat'],
            )

            loss = criterion(logits,b['y'])
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 5.0
            )

            opt.step()
            losses.append(float(loss.detach().cpu()))

        vp, vy = hg_predict(model,val_ds)
        val_ap = ap(vy,vp)

        if val_ap > best_ap + 1e-5:
            best_ap = val_ap
            best_state = {
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            bad = 0
        else:
            bad += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                arch_name,
                'fold',fold,
                'epoch',epoch,
                'loss',round(float(np.mean(losses)),4),
                'val_AP',round(val_ap,5),
                'best',round(best_ap,5),
                'alpha',round(
                    float(torch.sigmoid(model.alpha_logit).detach().cpu()),
                    4
                )
            )

        if bad >= HG_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model,best_ap,best_state

def run_hg_cv(arch_name):
    offsets = {}
    cursor = 0
    for r in train_df.itertuples():
        L = len(r.sequence)
        offsets[str(r.prot_id)] = (cursor,cursor+L)
        cursor += L

    oof = np.full(cursor,np.nan,dtype=np.float32)
    rows = []
    states = {}

    adir = HG_STATE_ROOT/arch_name
    adir.mkdir(parents=True,exist_ok=True)

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set = set(val_ids)

        tr_df = train_df[
            ~train_df.prot_id.isin(val_set)
        ].copy()

        va_df = train_df[
            train_df.prot_id.isin(val_set)
        ].copy()

        tr_ds = HGProteinDataset('DevSet1010', tr_df)
        va_ds = HGProteinDataset('DevSet1010', va_df)

        state_fp = adir/f'fold{fold}.pt'
        pred_fp = adir/f'fold{fold}_val_pred.npy'
        meta_fp = adir/f'fold{fold}_meta.json'

        expected_n = int(va_df.sequence.str.len().sum())
        reuse = False

        if state_fp.exists() and pred_fp.exists() and meta_fp.exists():
            try:
                meta = json.loads(meta_fp.read_text())
                pp = np.load(pred_fp)
                reuse = (
                    meta.get('architecture') == arch_name
                    and meta.get('representation') == BEST_REP_NAME
                    and meta.get('n_val') == expected_n
                    and pp.shape == (expected_n,)
                    and np.all(np.isfinite(pp))
                )
            except Exception:
                reuse = False

        if reuse:
            print(arch_name,'fold',fold,': REUSE')
            state = torch.load(state_fp,map_location='cpu')
            pred = np.load(pred_fp).astype(np.float32)

        else:
            print(arch_name,'fold',fold,': TRAIN')
            model,_,state = hg_train_fold(
                arch_name,tr_ds,va_ds,fold
            )

            pred,_ = hg_predict(model,va_ds)

            torch.save(state,state_fp)
            np.save(pred_fp,pred)
            meta_fp.write_text(json.dumps({
                'architecture':arch_name,
                'representation':BEST_REP_NAME,
                'blocks':BEST_REP_BLOCKS,
                'n_val':expected_n,
            },indent=2))

            del model
            torch.cuda.empty_cache()

        states[fold] = state

        pos = 0
        for r in va_df.itertuples():
            L = len(r.sequence)
            a,b = offsets[str(r.prot_id)]
            oof[a:b] = pred[pos:pos+L]
            pos += L

        yv = np.concatenate(
            va_df.metal_binding.apply(
                lambda x:np.asarray(x,dtype=np.float32)
            ).tolist()
        )

        rows.append({
            'architecture':arch_name,
            'fold':fold,
            'AUPRC':ap(yv,pred),
            'n_val_residues':len(pred),
            'n_val_positive':int(yv.sum()),
        })

    if not np.all(np.isfinite(oof)):
        raise RuntimeError(f'{arch_name}: incomplete hypergraph OOF')

    fold_df = pd.DataFrame(rows)
    yall = np.concatenate(
        train_df.metal_binding.apply(
            lambda x:np.asarray(x,dtype=np.float32)
        ).tolist()
    )

    summary = {
        'architecture':arch_name,
        'CV_AUPRC_mean':float(fold_df.AUPRC.mean()),
        'CV_AUPRC_std':float(fold_df.AUPRC.std(ddof=1)),
        'OOF_AUPRC':ap(yall,oof),
    }

    return summary,fold_df,oof,states

print('HG trainer: PASS')


In [ ]:
HG_RESULTS = {}
HG_FOLDS = []
HG_OOF = {}
HG_STATES = {}

if RUN_HYPERGRAPH_SCREEN:
    for arch in HG_FACTORY:
        print('\n'+'='*80)
        print('HYPERGRAPH:',arch)
        print('='*80)

        s,f,o,st = run_hg_cv(arch)

        HG_RESULTS[arch] = s
        HG_FOLDS.append(f)
        HG_OOF[arch] = o
        HG_STATES[arch] = st

hg_summary = pd.DataFrame(
    HG_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(drop=True)

hg_folds = pd.concat(
    HG_FOLDS,
    ignore_index=True
)

hg_summary.to_csv(RUN_ROOT/'hypergraph_cv.csv',index=False)
hg_folds.to_csv(RUN_ROOT/'hypergraph_folds.csv',index=False)

display(hg_summary)

BEST_HG = str(hg_summary.iloc[0].architecture)
BEST_HG_SCORE = float(hg_summary.iloc[0].CV_AUPRC_mean)

HG_MATCHED_LOCAL = {
    'HG_GatedResidual':'GatedFusion',
    'HG_AttentionResidual':'ModalityAttention',
    'HG_GatedMoE':'LatentMetalMoE',
}

matched_local = HG_MATCHED_LOCAL[BEST_HG]
matched_local_score = float(
    arch_summary.loc[
        arch_summary.architecture.eq(matched_local),
        'CV_AUPRC_mean'
    ].iloc[0]
)

HG_GAIN_MATCHED = BEST_HG_SCORE - matched_local_score

alphas = []
for fold,state in HG_STATES[BEST_HG].items():
    if 'alpha_logit' in state:
        alphas.append(
            float(torch.sigmoid(state['alpha_logit']).item())
        )

print('Best hypergraph:',BEST_HG)
print('Hypergraph CV AUPRC:',BEST_HG_SCORE)
print('Matched local:',matched_local,matched_local_score)
print('Hypergraph marginal gain:',HG_GAIN_MATCHED)
print('Hypergraph alpha by fold:',alphas)
print('Mean hypergraph alpha:',float(np.mean(alphas)) if alphas else None)

with open(RUN_ROOT/'hypergraph_decision.json','w') as f:
    json.dump({
        'best_hypergraph':BEST_HG,
        'best_hypergraph_cv_auprc':BEST_HG_SCORE,
        'matched_local_architecture':matched_local,
        'matched_local_cv_auprc':matched_local_score,
        'hypergraph_gain_vs_matched_local':HG_GAIN_MATCHED,
        'alpha_by_fold':alphas,
        'alpha_mean':float(np.mean(alphas)) if alphas else None,
    },f,indent=2)


# Global Dev-only lock

The final family is chosen before reading external results.

Candidates:
1. best locked custom multimodal model;
2. best existing residual residue graph;
3. best full site-hypergraph.

A more complex family replaces the simpler winner only if mean Dev CV AUPRC improves by at least `GLOBAL_COMPLEXITY_MARGIN`.


In [ ]:
custom_locked_score = float(
    arch_summary.loc[
        arch_summary.architecture.eq(LOCKED_ARCH),
        'CV_AUPRC_mean'
    ].iloc[0]
)

candidate_rows = [{
    'family':'CUSTOM',
    'name':LOCKED_ARCH,
    'CV_AUPRC':custom_locked_score,
    'complexity_rank':1,
}]

GRAPH_NAME_COL = None
GRAPH_SCORE = None
BEST_GRAPH_NAME = None

if graph_cv is not None and len(graph_cv):
    for c in ['representation','experiment','name','architecture']:
        if c in graph_cv.columns:
            GRAPH_NAME_COL = c
            break

    g = graph_cv.sort_values('CV_AUPRC',ascending=False).iloc[0]
    GRAPH_SCORE = float(g['CV_AUPRC'])
    BEST_GRAPH_NAME = (
        str(g[GRAPH_NAME_COL])
        if GRAPH_NAME_COL is not None
        else 'best_graph'
    )

    candidate_rows.append({
        'family':'GRAPH',
        'name':BEST_GRAPH_NAME,
        'CV_AUPRC':GRAPH_SCORE,
        'complexity_rank':2,
    })

candidate_rows.append({
    'family':'HYPERGRAPH',
    'name':BEST_HG,
    'CV_AUPRC':BEST_HG_SCORE,
    'complexity_rank':3,
})

global_candidates = pd.DataFrame(candidate_rows).sort_values(
    'complexity_rank'
).reset_index(drop=True)

winner = global_candidates.iloc[0].to_dict()

for _, row in global_candidates.iloc[1:].iterrows():
    gain = float(row.CV_AUPRC) - float(winner['CV_AUPRC'])
    if gain >= GLOBAL_COMPLEXITY_MARGIN:
        winner = row.to_dict()

GLOBAL_LOCK_FAMILY = str(winner['family'])
GLOBAL_LOCK_NAME = str(winner['name'])
GLOBAL_LOCK_SCORE = float(winner['CV_AUPRC'])

global_candidates['delta_vs_custom'] = (
    global_candidates.CV_AUPRC - custom_locked_score
)

display(global_candidates)

print('\nGLOBAL LOCK FAMILY:',GLOBAL_LOCK_FAMILY)
print('GLOBAL LOCK NAME:',GLOBAL_LOCK_NAME)
print('GLOBAL LOCK DEV AUPRC:',GLOBAL_LOCK_SCORE)

(RUN_ROOT/'GLOBAL_LOCK.json').write_text(
    json.dumps({
        'family':GLOBAL_LOCK_FAMILY,
        'name':GLOBAL_LOCK_NAME,
        'dev_cv_auprc':GLOBAL_LOCK_SCORE,
        'complexity_margin':GLOBAL_COMPLEXITY_MARGIN,
    },indent=2)
)


In [ ]:
def binary_metrics(y,p,threshold=0.5):
    y=np.asarray(y,dtype=int)
    p=np.asarray(p,dtype=float)
    pred=(p>=threshold).astype(int)

    return {
        'AUPRC':float(average_precision_score(y,p)),
        'AUROC':float(roc_auc_score(y,p)),
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'F1':float(f1_score(y,pred,zero_division=0)),
        'MCC':float(matthews_corrcoef(y,pred)),
    }

def choose_mcc_threshold(y,p):
    best=(-999,None)
    for t in np.linspace(0.01,0.99,199):
        m=matthews_corrcoef(y,(p>=t).astype(int))
        if m>best[0]:
            best=(m,float(t))
    return best[1],best[0]

locked_oof=ARCH_OOF[LOCKED_ARCH]
dev_y=PACKS['DevSet1010']['y'].astype(int)

OOF_THRESHOLD,OOF_MCC=choose_mcc_threshold(dev_y,locked_oof)

print('LOCKED_ARCH:',LOCKED_ARCH)
print('Dev OOF threshold maximizing MCC:',OOF_THRESHOLD)
print('Dev OOF MCC:',OOF_MCC)
print('Dev OOF @0.5:',binary_metrics(dev_y,locked_oof,0.5))
print('Dev OOF @OOF threshold:',binary_metrics(dev_y,locked_oof,OOF_THRESHOLD))


# External confirmation

After the global Dev lock, V4 evaluates the already Dev-selected winners from:
- custom multimodal family;
- residue graph family;
- hypergraph family.

The globally locked winner is the headline result.


In [ ]:
def predict_custom_external(split):
    pack = PACKS[split]
    ds = MultiArrayDataset(pack['mods'], pack['y'])
    fold_preds = []

    for fold,state in ARCH_STATES[LOCKED_ARCH].items():
        model = ARCH_FACTORY[LOCKED_ARCH](DIMS).to(DEVICE)
        model.load_state_dict(state)
        pp = predict_model(model, ds, indices=None)
        fold_preds.append(pp)
        del model
        torch.cuda.empty_cache()

    return np.mean(np.stack(fold_preds,axis=0),axis=0)

CUSTOM_EXTERNAL = {}
CUSTOM_ROWS = []

custom_oof = ARCH_OOF[LOCKED_ARCH]
dev_y = PACKS['DevSet1010']['y'].astype(int)

CUSTOM_THR, CUSTOM_OOF_MCC = choose_mcc_threshold(
    dev_y, custom_oof
)

for split in ['TestSet300','TestSetNew46']:
    p = predict_custom_external(split)
    y = PACKS[split]['y'].astype(int)
    CUSTOM_EXTERNAL[split] = p

    m05 = binary_metrics(y,p,0.5)
    moo = binary_metrics(y,p,CUSTOM_THR)

    row = {
        'family':'CUSTOM',
        'model':LOCKED_ARCH,
        'dataset':split,
        'Dev_CV_AUPRC':custom_locked_score,
        'Dev_OOF_threshold':CUSTOM_THR,
    }
    for k,v in m05.items():
        row[f'{k}_fixed0.5'] = v
    for k,v in moo.items():
        row[f'{k}_OOFthr'] = v

    CUSTOM_ROWS.append(row)

    pd.DataFrame({
        'prot_id':PACKS[split]['pids'],
        'residue_index':PACKS[split]['ridx'],
        'probability':p,
        'y_true':y,
    }).to_csv(
        RUN_ROOT/f'PRED_CUSTOM_{LOCKED_ARCH}_{split}.csv.gz',
        index=False
    )

custom_external_df = pd.DataFrame(CUSTOM_ROWS)
display(custom_external_df)


In [ ]:
def hg_predict_external(split, arch_name):
    ds = HGProteinDataset(split, SPLIT_DF[split])
    fold_preds = []

    for fold,state in HG_STATES[arch_name].items():
        model = HG_FACTORY[arch_name](DIMS).to(DEVICE)
        model.load_state_dict(state)
        pp,_ = hg_predict(model, ds)
        fold_preds.append(pp)
        del model
        torch.cuda.empty_cache()

    return np.mean(np.stack(fold_preds,axis=0),axis=0)

hg_oof = HG_OOF[BEST_HG]
HG_THR, HG_OOF_MCC = choose_mcc_threshold(dev_y, hg_oof)

HG_EXTERNAL = {}
HG_ROWS = []

for split in ['TestSet300','TestSetNew46']:
    p = hg_predict_external(split, BEST_HG)
    y = np.concatenate(
        SPLIT_DF[split].metal_binding.apply(
            lambda x: np.asarray(x,dtype=int)
        ).tolist()
    )

    HG_EXTERNAL[split] = p

    m05 = binary_metrics(y,p,0.5)
    moo = binary_metrics(y,p,HG_THR)

    row = {
        'family':'HYPERGRAPH',
        'model':BEST_HG,
        'dataset':split,
        'Dev_CV_AUPRC':BEST_HG_SCORE,
        'Dev_OOF_threshold':HG_THR,
        'mean_hypergraph_alpha':float(np.mean(alphas)) if alphas else np.nan,
    }

    for k,v in m05.items():
        row[f'{k}_fixed0.5'] = v
    for k,v in moo.items():
        row[f'{k}_OOFthr'] = v

    HG_ROWS.append(row)

    pids, ridx = [], []
    for r in SPLIT_DF[split].itertuples():
        pids.extend([str(r.prot_id)]*len(r.sequence))
        ridx.extend(range(len(r.sequence)))

    pd.DataFrame({
        'prot_id':pids,
        'residue_index':ridx,
        'probability':p,
        'y_true':y,
    }).to_csv(
        RUN_ROOT/f'PRED_HYPERGRAPH_{BEST_HG}_{split}.csv.gz',
        index=False
    )

hg_external_df = pd.DataFrame(HG_ROWS)
display(hg_external_df)


In [ ]:
graph_external_df = pd.DataFrame()

if graph_ext is not None and len(graph_ext):
    q = graph_ext.copy()

    name_col = None
    for c in ['representation','experiment','name','architecture']:
        if c in q.columns:
            name_col = c
            break

    if name_col is not None and BEST_GRAPH_NAME is not None:
        q = q[q[name_col].astype(str).eq(BEST_GRAPH_NAME)].copy()

    q['family'] = 'GRAPH'
    q['Dev_CV_AUPRC'] = GRAPH_SCORE
    graph_external_df = q

    display(graph_external_df)
else:
    print('No graph external table was produced.')


# Locked baseline comparison

No baseline inference is rerun.

The completed metal-only rows for bindEmbed21DL, bindNode24, ESMBind, and MetalBind are read from the frozen baseline reproduction directory.


In [ ]:
BASE_RES = BASELINE_WORK/'results'

baseline_files = [
    BASE_RES/'FULL_DIRECT_BASELINE_METRICS_300_46.csv',
    BASE_RES/'ESMBind_TRANSFER_FULL_METRICS_300_46.csv',
    BASE_RES/'MetalBind_TRANSFER_FULL_METRICS_300_46.csv',
]

frames = []

for fp in baseline_files:
    if not fp.exists():
        print('Missing:',fp)
        continue

    z = pd.read_csv(fp)

    if 'operating_point' in z.columns:
        z = z[z.operating_point.eq('fixed_0.5')].copy()

    if 'label' in z.columns:
        z = z[z.label.eq('metal_binding')].copy()

    frames.append(z)

baselines = pd.concat(frames,ignore_index=True,sort=False)

baseline_cols = [
    c for c in [
        'method','dataset','label','AP_AUPRC','AUPRC','AUROC',
        'global_positive_F1','global_MCC',
        'protein_mean_F1','protein_mean_MCC'
    ]
    if c in baselines.columns
]

display(baselines[baseline_cols])

baselines.to_csv(
    RUN_ROOT/'locked_baseline_metal_rows.csv',
    index=False
)


# Automatic scientific insight report

This section decomposes where performance gain came from:
representation, ESM-IF, site context, multimodal fusion, generic graph, and full site-hypergraph.

It also reports candidate-site coverage and the learned hypergraph residual strength.


In [ ]:
def _rep_score(name):
    x = rep_cv.loc[
        rep_cv.representation.eq(name),
        'CV_AUPRC'
    ]
    return float(x.iloc[0]) if len(x) else np.nan

score_anchor = _rep_score('Anchor_ESM_ATOM_CHEM_MULTI2')
score_esmif = _rep_score('Anchor_plus_ESMIF')
score_sitectx = _rep_score('Anchor_plus_SITECTX')
score_both = _rep_score('Anchor_plus_ESMIF_SITECTX')

concat_custom_score = float(
    arch_summary.loc[
        arch_summary.architecture.eq('ConcatMLP'),
        'CV_AUPRC_mean'
    ].iloc[0]
)

gain_rows = [
    {
        'stage':'Historical standardized compact MLP',
        'model':'Anchor ESM+ATOM+CHEM+MULTI2',
        'Dev_CV_AUPRC':score_anchor,
        'delta_vs_anchor':0.0,
    },
    {
        'stage':'Representation',
        'model':'Anchor + ESMIF',
        'Dev_CV_AUPRC':score_esmif,
        'delta_vs_anchor':score_esmif-score_anchor,
    },
    {
        'stage':'Representation',
        'model':'Anchor + SITECTX32',
        'Dev_CV_AUPRC':score_sitectx,
        'delta_vs_anchor':score_sitectx-score_anchor,
    },
    {
        'stage':'Representation',
        'model':'Anchor + ESMIF + SITECTX32',
        'Dev_CV_AUPRC':score_both,
        'delta_vs_anchor':score_both-score_anchor,
    },
    {
        'stage':'Locked representation / concat control',
        'model':'ConcatMLP',
        'Dev_CV_AUPRC':concat_custom_score,
        'delta_vs_anchor':concat_custom_score-score_anchor,
    },
    {
        'stage':'Best multimodal local',
        'model':LOCKED_ARCH,
        'Dev_CV_AUPRC':custom_locked_score,
        'delta_vs_anchor':custom_locked_score-score_anchor,
    },
]

if GRAPH_SCORE is not None:
    gain_rows.append({
        'stage':'Best residue graph',
        'model':BEST_GRAPH_NAME,
        'Dev_CV_AUPRC':GRAPH_SCORE,
        'delta_vs_anchor':GRAPH_SCORE-score_anchor,
    })

gain_rows.append({
    'stage':'Best site hypergraph',
    'model':BEST_HG,
    'Dev_CV_AUPRC':BEST_HG_SCORE,
    'delta_vs_anchor':BEST_HG_SCORE-score_anchor,
})

gain_ladder = pd.DataFrame(gain_rows).sort_values(
    'Dev_CV_AUPRC',
    ascending=False
)

display(gain_ladder)

gain_ladder.to_csv(
    RUN_ROOT/'INSIGHT_gain_ladder.csv',
    index=False
)

insight = {
    'anchor_cv_auprc': score_anchor,
    'esmif_gain': (
        score_esmif-score_anchor
        if np.isfinite(score_esmif) else None
    ),
    'sitectx_gain': (
        score_sitectx-score_anchor
        if np.isfinite(score_sitectx) else None
    ),
    'esmif_plus_sitectx_gain': (
        score_both-score_anchor
        if np.isfinite(score_both) else None
    ),
    'best_representation': BEST_REP_NAME,
    'best_representation_gain': BEST_REP_GAIN,
    'locked_custom_architecture': LOCKED_ARCH,
    'custom_fusion_gain_vs_concat': (
        custom_locked_score-concat_custom_score
    ),
    'best_graph': BEST_GRAPH_NAME,
    'graph_gain_vs_anchor': (
        GRAPH_SCORE-score_anchor
        if GRAPH_SCORE is not None else None
    ),
    'best_hypergraph': BEST_HG,
    'hypergraph_gain_vs_matched_local': HG_GAIN_MATCHED,
    'hypergraph_alpha_mean': (
        float(np.mean(alphas)) if alphas else None
    ),
    'global_lock_family': GLOBAL_LOCK_FAMILY,
    'global_lock_name': GLOBAL_LOCK_NAME,
    'global_lock_dev_auprc': GLOBAL_LOCK_SCORE,
}

(RUN_ROOT/'INSIGHT_hypothesis_summary.json').write_text(
    json.dumps(insight,indent=2)
)

print(json.dumps(insight,indent=2))


In [ ]:
def verdict(gain, margin=MIN_MEAN_AUPRC_GAIN):
    if gain is None or not np.isfinite(gain):
        return 'NOT_EVALUABLE'
    if gain >= 0.01:
        return 'STRONG'
    if gain >= 0.005:
        return 'CLEAR'
    if gain >= margin:
        return 'MODEST_BUT_REAL'
    if gain > 0:
        return 'WEAK'
    return 'NO_GAIN'

site_candidates = [
    x for x in [
        insight['sitectx_gain'],
        insight['esmif_plus_sitectx_gain']
    ]
    if x is not None and np.isfinite(x)
]
site_gain = max(site_candidates) if site_candidates else None

verdict_rows = [
    {
        'hypothesis':
            'ESM-IF adds learned structural information beyond ESM+chemistry',
        'gain':insight['esmif_gain'],
        'verdict':verdict(insight['esmif_gain']),
    },
    {
        'hypothesis':
            'Higher-order candidate-site context contains marginal signal',
        'gain':site_gain,
        'verdict':verdict(site_gain),
    },
    {
        'hypothesis':
            'Learned multimodal fusion beats simple concatenation',
        'gain':insight['custom_fusion_gain_vs_concat'],
        'verdict':verdict(insight['custom_fusion_gain_vs_concat']),
    },
    {
        'hypothesis':
            'Generic residue graph adds useful signal',
        'gain':insight['graph_gain_vs_anchor'],
        'verdict':verdict(insight['graph_gain_vs_anchor']),
    },
    {
        'hypothesis':
            'Full candidate-site hypergraph adds signal beyond its matched local model',
        'gain':insight['hypergraph_gain_vs_matched_local'],
        'verdict':verdict(insight['hypergraph_gain_vs_matched_local']),
    },
]

verdict_df = pd.DataFrame(verdict_rows)
display(verdict_df)

verdict_df.to_csv(
    RUN_ROOT/'INSIGHT_hypothesis_verdicts.csv',
    index=False
)


In [ ]:
family_external_df = pd.concat(
    [custom_external_df, hg_external_df],
    ignore_index=True,
    sort=False
)

family_external_df.to_csv(
    RUN_ROOT/'EXTERNAL_family_winners_custom_hypergraph.csv',
    index=False
)

print('CUSTOM + HYPERGRAPH FAMILY WINNERS')
display(family_external_df)

if len(graph_external_df):
    print('GRAPH FAMILY WINNER')
    display(graph_external_df)

print('\nGLOBAL HEADLINE MODEL')
print(
    GLOBAL_LOCK_FAMILY,
    GLOBAL_LOCK_NAME,
    'Dev CV AUPRC =',
    GLOBAL_LOCK_SCORE
)

if GLOBAL_LOCK_FAMILY == 'CUSTOM':
    headline = custom_external_df.copy()
elif GLOBAL_LOCK_FAMILY == 'HYPERGRAPH':
    headline = hg_external_df.copy()
elif GLOBAL_LOCK_FAMILY == 'GRAPH':
    headline = graph_external_df.copy()
else:
    raise RuntimeError(GLOBAL_LOCK_FAMILY)

headline.to_csv(
    RUN_ROOT/'FINAL_HEADLINE_EXTERNAL.csv',
    index=False
)

display(headline)


In [ ]:
print('='*96)
print('GEAR METAL V4 — SCIENTIFIC INTERPRETATION')
print('='*96)

print(f'\nHistorical compact anchor Dev CV AUPRC: {score_anchor:.4f}')
print(
    f'Best representation: {BEST_REP_NAME} '
    f'({BEST_REP_GAIN:+.4f} vs anchor)'
)
print(
    f'Best local multimodal architecture: {LOCKED_ARCH} '
    f'({custom_locked_score:.4f})'
)

if GRAPH_SCORE is not None:
    print(
        f'Best residue graph: {BEST_GRAPH_NAME} '
        f'({GRAPH_SCORE:.4f})'
    )

print(
    f'Best site hypergraph: {BEST_HG} '
    f'({BEST_HG_SCORE:.4f}; '
    f'{HG_GAIN_MATCHED:+.4f} vs matched local)'
)

if alphas:
    print(
        f'Learned mean hypergraph residual weight: '
        f'{np.mean(alphas):.4f}'
    )

print(
    f'\nGLOBAL DEV-LOCKED WINNER: '
    f'{GLOBAL_LOCK_FAMILY} / {GLOBAL_LOCK_NAME} '
    f'({GLOBAL_LOCK_SCORE:.4f})'
)

if verdict(insight['esmif_gain']) in {
    'STRONG','CLEAR','MODEST_BUT_REAL'
}:
    print(
        '✓ ESM-IF contributes complementary learned 3D information; retain it.'
    )
else:
    print(
        '• ESM-IF marginal gain is weak; it may not justify mandatory use.'
    )

if verdict(insight['custom_fusion_gain_vs_concat']) in {
    'STRONG','CLEAR','MODEST_BUT_REAL'
}:
    print(
        '✓ Modality-specific projection/fusion matters; raw concatenation is suboptimal.'
    )
else:
    print(
        '• Fusion gain is small; prefer the simpler concat architecture.'
    )

if insight['graph_gain_vs_anchor'] is not None:
    if verdict(insight['graph_gain_vs_anchor']) in {
        'STRONG','CLEAR','MODEST_BUT_REAL'
    }:
        print('✓ Residue-level graph relations add useful signal.')
    else:
        print(
            '• Generic residue graph is weak/redundant with ATOM/CHEM.'
        )

if verdict(insight['hypergraph_gain_vs_matched_local']) in {
    'STRONG','CLEAR','MODEST_BUT_REAL'
}:
    print(
        '✓ Candidate coordination-site message passing adds information '
        'beyond the matched local encoder.'
    )
else:
    print(
        '• Site hypergraph does not clearly beat its matched local encoder.'
    )

if BEST_HG == 'HG_GatedMoE' or LOCKED_ARCH == 'LatentMetalMoE':
    print(
        '✓ Latent metal experts are supported, suggesting heterogeneous metal chemistry.'
    )

print('\nSite-hypergraph coverage:')
display(
    hg_audit.groupby('dataset').agg(
        mean_candidate_sites=('n_candidate_sites','mean'),
        median_candidate_sites=('n_candidate_sites','median'),
        mean_site_covered_fraction=('site_covered_fraction','mean'),
    ).reset_index()
)

print('\nFinal artifact directory:')
print(RUN_ROOT)
print('='*96)


In [ ]:
manifest = {
    'gear_version': GEAR_VERSION,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),

    'development_set':
        'DevSet1010 official protein-level 5-fold split',

    'external_sets': [
        'TestSet300',
        'TestSetNew46'
    ],

    'primary_selection_metric':
        'residue-level CV AUPRC',

    'historical_anchor_blocks':
        ANCHOR_BLOCKS,

    'best_representation':
        BEST_REP_NAME,

    'best_representation_blocks':
        BEST_REP_BLOCKS,

    'locked_custom_architecture':
        LOCKED_ARCH,

    'best_graph':
        BEST_GRAPH_NAME,

    'best_hypergraph':
        BEST_HG,

    'global_lock_family':
        GLOBAL_LOCK_FAMILY,

    'global_lock_name':
        GLOBAL_LOCK_NAME,

    'global_lock_dev_auprc':
        GLOBAL_LOCK_SCORE,

    'complexity_margin':
        GLOBAL_COMPLEXITY_MARGIN,

    'hypergraph_radius_A':
        HG_RADIUS_A,

    'hypergraph_max_donors':
        HG_MAX_DONORS,

    'hypergraph_alpha_mean':
        float(np.mean(alphas))
        if alphas else None,

    'insight':
        insight,
}

(RUN_ROOT/'FINAL_V4_MANIFEST.json').write_text(
    json.dumps(manifest,indent=2)
)

print(json.dumps(manifest,indent=2))

print('\n'+'='*96)
print('GEAR METAL V4 FULL INSIGHT RUN COMPLETE')
print('='*96)
print(
    'Representation → Fusion → Residue Graph → Full Site Hypergraph '
    'were all tested under Dev-first selection.'
)
print('Headline family:', GLOBAL_LOCK_FAMILY)
print('Headline model:', GLOBAL_LOCK_NAME)
print('Dev CV AUPRC:', GLOBAL_LOCK_SCORE)
print('Results saved to:', RUN_ROOT)
print('='*96)
